# 🛒 E-Commerce Product Recommendation System

**Models:** LightFM (BPR) · SVD (Matrix Factorisation) · KNN (Collaborative Filtering)

**Datasets:** orders · products · reviews · users · carts · interactions

---
> **Evaluation:** With 100 products and ~7.9 interactions/user we use **rating-prediction accuracy**
> (RMSE Acc, MAE Acc) and **ranking metrics** (NDCG@20, Hit Rate@20). All targets ≥ 0.80.

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split as sk_split
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('All libraries loaded')

ModuleNotFoundError: No module named 'surprise'

## 2. Load Data

In [3]:
orders       = pd.read_csv('data/orders.csv')
products     = pd.read_csv('data/products.csv')
reviews      = pd.read_csv('data/reviews.csv')
users        = pd.read_csv('data/users.csv')
carts        = pd.read_csv('data/carts.csv')
interactions = pd.read_csv('data/interactions.csv')

for name, df in [('orders',orders),('products',products),('reviews',reviews),
                 ('users',users),('carts',carts),('interactions',interactions)]:
    print(f'{name:15s}: {df.shape[0]:>6,} rows x {df.shape[1]} cols')

print(f'\nProducts in catalog : {products.shape[0]}')
print(f'Unique users        : {reviews.user_id.nunique():,}')

orders         : 48,622 rows x 6 cols
products       :    100 rows x 8 cols
reviews        : 66,776 rows x 8 cols
users          : 10,000 rows x 9 cols
carts          : 12,246 rows x 7 cols
interactions   :  5,000 rows x 4 cols

Products in catalog : 100
Unique users        : 9,542


## 3. Build Interaction Matrix

Fuse **explicit** review ratings (1-5 stars) with **implicit** cart-add signals.
Both are normalised to [0, 1] and we keep the maximum signal per (user, product).

In [4]:
# Integer ID mappings
all_users    = pd.concat([reviews['user_id'], carts['user_id']]).unique()
all_products = pd.concat([reviews['product_id'], carts['product_id']]).unique()
uid_map  = {u: i for i, u in enumerate(sorted(all_users))}
pid_map  = {p: i for i, p in enumerate(sorted(all_products))}
rev_uid  = {v: k for k, v in uid_map.items()}
rev_pid  = {v: k for k, v in pid_map.items()}
n_users, n_items = len(uid_map), len(pid_map)
print(f'Users: {n_users:,}  |  Products: {n_items}')

# Explicit signals: star ratings
rv = reviews[reviews['user_id'].isin(uid_map) & reviews['product_id'].isin(pid_map)].copy()
rv['u'] = rv['user_id'].map(uid_map)
rv['p'] = rv['product_id'].map(pid_map)
rv['score'] = ((rv['rating'] - 1) / 4.0
               + 0.05 * rv['verified_purchase'].astype(float)).clip(0, 1)

# Implicit signals: cart adds
ct = carts[carts['user_id'].isin(uid_map) & carts['product_id'].isin(pid_map)].copy()
ct['u'] = ct['user_id'].map(uid_map)
ct['p'] = ct['product_id'].map(pid_map)
ct['score'] = 0.65

# Merge: max signal per (user, product)
all_signals = (pd.concat([rv[['u','p','score']], ct[['u','p','score']]])
               .groupby(['u','p'], as_index=False)['score'].max())

print(f'Total interactions  : {len(all_signals):,}')
print(f'Avg per user        : {all_signals.groupby("u").size().mean():.1f}')
print(f'Catalog density     : {len(all_signals)/(n_users*n_items)*100:.1f}%')

Users: 9,699  |  Products: 100
Total interactions  : 77,032
Avg per user        : 7.9
Catalog density     : 7.9%


## 4. Train / Test Split

In [5]:
train_df, test_df = sk_split(all_signals, test_size=0.2, random_state=42)

def build_sparse(df):
    return sp.csr_matrix(
        (df['score'].values, (df['u'].values, df['p'].values)),
        shape=(n_users, n_items))

train_mat = build_sparse(train_df)
test_mat  = build_sparse(test_df)
print(f'Train: {train_mat.nnz:,}  |  Test: {test_mat.nnz:,} interactions')

Train: 61,625  |  Test: 15,407 interactions


## 5. Evaluation Metrics

| Metric | Description | Target |
|---|---|---|
| **RMSE Accuracy** | `1 - RMSE` on [0,1] scores | >= 0.80 |
| **MAE Accuracy** | `1 - MAE` on [0,1] scores | >= 0.80 |
| **NDCG@20** | Ranking quality (relevance-weighted) | >= 0.80 |
| **Hit Rate@20** | >= 1 relevant item in top-20 | >= 0.80 |

In [6]:
def rmse_accuracy(pred, actual):
    return 1.0 - np.sqrt(np.mean((pred - actual) ** 2))

def mae_accuracy(pred, actual):
    return 1.0 - np.mean(np.abs(pred - actual))

def ndcg_at_k(score_matrix, test_matrix, k=20):
    td       = test_matrix.toarray()
    true_rel = (td >= 0.5).astype(float)
    mask     = true_rel.sum(axis=1) > 0
    if mask.sum() == 0: return 0.0
    s = np.where(np.isfinite(score_matrix[mask]), score_matrix[mask], 0.0)
    return ndcg_score(true_rel[mask], s.astype(np.float64), k=k)

def hit_rate_at_k(score_matrix, test_matrix, k=20, threshold=0.5):
    hits = total = 0
    for u in range(test_matrix.shape[0]):
        true_items = set(test_matrix[u].indices[test_matrix[u].data >= threshold])
        if not true_items: continue
        top_k = set(np.argsort(score_matrix[u])[::-1][:k])
        hits += int(bool(top_k & true_items))
        total += 1
    return hits / total if total > 0 else 0.0

def evaluate(name, score_matrix, train_matrix, test_matrix, k=20):
    # Rating-prediction accuracy on held-out pairs
    rows, cols = test_matrix.nonzero()
    actual     = test_matrix.data
    predicted  = score_matrix[rows, cols].clip(0, 1)
    rmse_acc   = rmse_accuracy(predicted, actual)
    mae_acc    = mae_accuracy(predicted, actual)

    # Ranking metrics: hide training items
    rank_scores = score_matrix.copy()
    rank_scores[train_matrix.nonzero()] = -1e9
    ndcg_val = ndcg_at_k(rank_scores, test_matrix, k=k)
    hr_val   = hit_rate_at_k(rank_scores, test_matrix, k=k)

    print(f'\n{name}')
    print(f'  RMSE Accuracy : {rmse_acc:.4f}  {"OK" if rmse_acc >= 0.8 else "below target"}')
    print(f'  MAE Accuracy  : {mae_acc:.4f}  {"OK" if mae_acc  >= 0.8 else "below target"}')
    print(f'  NDCG@{k}       : {ndcg_val:.4f}  {"OK" if ndcg_val >= 0.8 else "below target"}')
    print(f'  Hit Rate@{k}   : {hr_val:.4f}  {"OK" if hr_val   >= 0.8 else "below target"}')
    return {'Model': name, 'RMSE_Acc': rmse_acc, 'MAE_Acc': mae_acc,
            f'NDCG@{k}': ndcg_val, f'HitRate@{k}': hr_val}

results = []
print('Evaluation helpers ready')

Evaluation helpers ready


---
## 6. Model 1 — LightFM (BPR)

**Bayesian Personalised Ranking** learns latent user/item embeddings by maximising
the predicted score gap between a liked item `i` and a random negative `j`.

> Note: LightFM's C extension does not compile on Python 3.12 (known upstream issue).
> This cell implements the **identical BPR algorithm** with the same `fit/predict` API.
> On Python 3.10/3.11 you can swap this class for `from lightfm import LightFM`.

In [7]:
class LightFM_BPR:
    """
    Bayesian Personalised Ranking Matrix Factorisation.
    Mirrors LightFM's API: fit(interactions_csr) -> predict().
    """
    def __init__(self, n_components=64, learning_rate=0.05,
                 n_epochs=50, reg=1e-4, neg_ratio=5):
        self.k, self.lr    = n_components, learning_rate
        self.epochs        = n_epochs
        self.reg           = reg
        self.neg_ratio     = neg_ratio

    def fit(self, interactions_csr):
        n_users, n_items = interactions_csr.shape
        scale   = np.sqrt(2.0 / (n_users + self.k))      # Xavier init
        self.U  = np.random.randn(n_users, self.k).astype(np.float32) * scale
        self.V  = np.random.randn(n_items, self.k).astype(np.float32) * scale
        self.b  = np.zeros(n_items, dtype=np.float32)     # item bias

        coo = interactions_csr.tocoo()
        u_pos, i_pos = coo.row.copy(), coo.col.copy()

        for epoch in range(self.epochs):
            perm = np.random.permutation(len(u_pos))
            u_pos, i_pos = u_pos[perm], i_pos[perm]

            for u, i in zip(u_pos, i_pos):
                for _ in range(self.neg_ratio):
                    j     = np.random.randint(n_items)
                    x_uij = (self.U[u] @ self.V[i] + self.b[i]) - \
                            (self.U[u] @ self.V[j] + self.b[j])
                    g     = -1.0 / (1.0 + np.exp(x_uij))   # sigmoid gradient

                    self.U[u] -= self.lr * (g * (self.V[i] - self.V[j]) + self.reg * self.U[u])
                    self.V[i] -= self.lr * ( g * self.U[u] + self.reg * self.V[i])
                    self.V[j] -= self.lr * (-g * self.U[u] + self.reg * self.V[j])
                    self.b[i] -= self.lr * ( g  + self.reg * self.b[i])
                    self.b[j] -= self.lr * (-g  + self.reg * self.b[j])

            if (epoch + 1) % 10 == 0:
                print(f'  Epoch {epoch+1:>3}/{self.epochs}')
        return self

    def predict(self):
        """Full [n_users x n_items] score matrix."""
        return (self.U @ self.V.T + self.b).astype(np.float64)

In [8]:
print('Training LightFM BPR ...')
lightfm = LightFM_BPR(n_components=64, learning_rate=0.05,
                       n_epochs=50, reg=1e-4, neg_ratio=5)
lightfm.fit(train_mat)

# Sigmoid-normalise to [0,1] so RMSE/MAE are on the same scale as targets
raw       = lightfm.predict()
lf_scores = 1.0 / (1.0 + np.exp(-raw))

res = evaluate('LightFM (BPR)', lf_scores, train_mat, test_mat)
results.append(res)

Training LightFM BPR ...


C:\Users\hp\AppData\Local\Temp\ipykernel_23100\1859147118.py:32: RuntimeWarning: overflow encountered in exp
  g     = -1.0 / (1.0 + np.exp(x_uij))   # sigmoid gradient


  Epoch  10/50
  Epoch  20/50
  Epoch  30/50
  Epoch  40/50
  Epoch  50/50


C:\Users\hp\AppData\Local\Temp\ipykernel_23100\2649824993.py:8: RuntimeWarning: overflow encountered in exp
  lf_scores = 1.0 / (1.0 + np.exp(-raw))



LightFM (BPR)
  RMSE Accuracy : 0.4359  below target
  MAE Accuracy  : 0.5300  below target
  NDCG@20       : 0.0947  below target
  Hit Rate@20   : 0.2927  below target


---
## 7. Model 2 — SVD (Matrix Factorisation)

Classic **Funk SVD** via `scikit-surprise`. Decomposes the user-item matrix
into latent factor matrices with bias terms. Excellent at rating prediction.

In [9]:
# Build Surprise dataset from training interactions
svd_train           = train_df.copy()
svd_train['rating'] = svd_train['score'] * 4 + 1   # rescale 0-1 -> 1-5

reader   = Reader(rating_scale=(1, 5))
dataset  = Dataset.load_from_df(svd_train[['u', 'p', 'rating']], reader)
trainset, _ = surprise_split(dataset, test_size=0.05, random_state=42)

print('Training SVD ...')
svd = SVD(n_factors=150, n_epochs=40, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

print('Building full score matrix ...')
svd_scores = np.zeros((n_users, n_items), dtype=np.float64)
for u in range(n_users):
    for p in range(n_items):
        svd_scores[u, p] = svd.predict(u, p).est

svd_scores = (svd_scores - 1) / 4.0   # normalise back to 0-1

res = evaluate('SVD (Matrix Factorisation)', svd_scores, train_mat, test_mat)
results.append(res)

NameError: name 'Reader' is not defined

---
## 8. Model 3 — KNN (User-Based Collaborative Filtering)

Finds the **K most similar users** (cosine similarity) then takes a
weighted average of their known ratings to predict unseen items.

In [10]:
print('Training KNN (User-Based CF) ...')
knn_model = NearestNeighbors(metric='cosine', algorithm='brute',
                              n_neighbors=20, n_jobs=-1)
knn_model.fit(train_mat)
distances, indices = knn_model.kneighbors(train_mat)

item_means = train_mat.mean(axis=0).A1   # global fallback for cold users
knn_scores = np.zeros((n_users, n_items), dtype=np.float64)

for u in range(n_users):
    nb_idx = indices[u][1:]                         # exclude self
    sim    = np.maximum(1 - distances[u][1:], 0)   # cosine -> similarity

    if sim.sum() == 0:
        knn_scores[u] = item_means
        continue

    nb_mat        = train_mat[nb_idx].toarray()    # (K x n_items)
    w             = sim[:, np.newaxis]             # (K x 1)
    knn_scores[u] = (w * nb_mat).sum(0) / (w.sum() + 1e-9)

res = evaluate('KNN (User-Based CF)', knn_scores, train_mat, test_mat)
results.append(res)
print('\nAll models trained')

Training KNN (User-Based CF) ...

KNN (User-Based CF)
  RMSE Accuracy : 0.2389  below target
  MAE Accuracy  : 0.2576  below target
  NDCG@20       : 0.1351  below target
  Hit Rate@20   : 0.4682  below target

All models trained


---
## 9. Results Summary

In [11]:
summary = pd.DataFrame(results).set_index('Model')

print('='*65)
print('         MODEL COMPARISON  (target >= 0.80 on all metrics)')
print('='*65)
print(summary.applymap(lambda x: f'{x:.4f}').to_string())
print('='*65)

print('\nMetric breakdown:')
for col in summary.columns:
    for model in summary.index:
        val  = summary.loc[model, col]
        flag = 'OK' if val >= 0.80 else 'BELOW'
        print(f'  [{flag:5s}] {model:38s} {col}: {val:.4f}')

         MODEL COMPARISON  (target >= 0.80 on all metrics)


AttributeError: 'DataFrame' object has no attribute 'applymap'

---
## 10. Recommendation Function (Web App Ready)

In [ ]:
def get_recommendations(user_id_str: str, model: str = 'svd', top_k: int = 10) -> pd.DataFrame:
    """
    Return top-K product recommendations for a user.

    Parameters
    ----------
    user_id_str : str   e.g. 'user_00042'
    model       : str   'lightfm' | 'svd' | 'knn'
    top_k       : int   number of results

    Returns
    -------
    pd.DataFrame  product_id | name | category | price | avg_rating | rec_score
    """
    # Cold-start: return popular items for unknown users
    if user_id_str not in uid_map:
        print('Unknown user - returning popular items.')
        return (products.sort_values('review_count', ascending=False)
                        .head(top_k)[['product_id','name','category','price','avg_rating']])

    u = uid_map[user_id_str]
    score_lookup = {'lightfm': lf_scores, 'svd': svd_scores, 'knn': knn_scores}
    if model not in score_lookup:
        raise ValueError(f'model must be one of {list(score_lookup)}')

    scores = score_lookup[model][u].copy()
    scores[train_mat[u].indices] = -1e9   # hide already-seen items

    top_items  = np.argsort(scores)[::-1][:top_k]
    top_pids   = [rev_pid[i] for i in top_items]
    top_scores = scores[top_items]

    return (products[products['product_id'].isin(top_pids)]
            .merge(pd.DataFrame({'product_id': top_pids, 'rec_score': top_scores}), on='product_id')
            .sort_values('rec_score', ascending=False)
            [['product_id','name','category','price','avg_rating','rec_score']]
            .reset_index(drop=True))


# Demo
sample_user = list(uid_map.keys())[10]
print(f'Top-10 recommendations for: {sample_user}\n')
print(get_recommendations(sample_user, model='svd').to_string(index=False))

In [ ]:
# Compare all three models side-by-side for the same user
for m in ['lightfm', 'svd', 'knn']:
    recs = get_recommendations(sample_user, model=m)
    print(f'\n--- {m.upper()} ---')
    print(recs[['name','category','price','rec_score']].to_string(index=False))